In [ ]:
import shutil
from pathlib import Path
from tqdm import tqdm

# =========================================================
# PATHS
# =========================================================
DATASET_ROOT = Path("/kaggle/input/datasets/ahmedshayan1/dataset-3/Dataset_3")
DEST_DIR = Path("/kaggle/working/Kidney/normal")

# YOLO split folders
SPLITS = ["train", "valid", "test"]

# Class ID for "normal kidney" from:
# names: ['kidney-stone', 'normal kidney']
NORMAL_CLASS_ID = 1

# Valid image extensions
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# =========================================================
# CREATE DESTINATION
# =========================================================
DEST_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# HELPER: find matching image for a label file
# =========================================================
def find_image_for_label(label_path: Path, images_dir: Path):
    for ext in IMAGE_EXTS:
        img_path = images_dir / f"{label_path.stem}{ext}"
        if img_path.exists():
            return img_path
    return None

# =========================================================
# COPY ONLY NORMAL IMAGES
# =========================================================
copied = 0
skipped = 0

for split in SPLITS:
    labels_dir = DATASET_ROOT / split / "labels"
    images_dir = DATASET_ROOT / split / "images"

    if not labels_dir.exists() or not images_dir.exists():
        print(f"Skipping missing split: {split}")
        continue

    label_files = list(labels_dir.glob("*.txt"))

    for label_file in tqdm(label_files, desc=f"Processing {split}"):
        try:
            lines = label_file.read_text().strip().splitlines()
        except Exception:
            skipped += 1
            continue

        if not lines:
            continue

        class_ids = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 1:
                try:
                    class_ids.append(int(parts[0]))
                except:
                    pass

        # Keep only images where ALL labels are normal kidney
        if class_ids and all(cls_id == NORMAL_CLASS_ID for cls_id in class_ids):
            img_path = find_image_for_label(label_file, images_dir)

            if img_path is not None:
                dst_path = DEST_DIR / f"{split}_{img_path.name}"
                shutil.copy2(img_path, dst_path)
                copied += 1
            else:
                skipped += 1

print("\nDone.")
print(f"Normal images copied: {copied}")
print(f"Skipped files: {skipped}")
print(f"Saved to: {DEST_DIR}")

In [ ]:
!ls

In [ ]:
# !rm -rf Kidney

In [ ]:
import random
import shutil
from pathlib import Path
from tqdm import tqdm

# =========================================================
# PATHS
# =========================================================
NORMAL_SRC = Path("/kaggle/working/Kidney/normal")

DISEASE_ROOT = Path(
    "/kaggle/input/datasets/ahmedshayan1/dataset-multicalasess/"
    "KidneyDiseaseClassification.folder/train"
)

CYSTS_DIR = DISEASE_ROOT / "cysts"
STONES_DIR = DISEASE_ROOT / "stones"
TUMORS_DIR = DISEASE_ROOT / "tumors"

NORMAL_DEST = Path("/kaggle/working/Kidney/final_normal")
DISEASE_DEST = Path("/kaggle/working/Kidney/disease")

# =========================================================
# SETTINGS
# =========================================================
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
SEED = 42
random.seed(SEED)

# =========================================================
# HELPERS
# =========================================================
def get_images(folder: Path):
    return [p for p in folder.rglob("*") if p.suffix.lower() in IMAGE_EXTS]

def clean_folder(folder: Path):
    folder.mkdir(parents=True, exist_ok=True)
    for f in folder.iterdir():
        if f.is_file():
            f.unlink()

def copy_images(files, dest: Path, prefix="img"):
    dest.mkdir(parents=True, exist_ok=True)
    for i, src in enumerate(tqdm(files, desc=f"Copying to {dest.name}"), start=1):
        dst = dest / f"{prefix}_{i}_{src.name}"
        counter = 1
        while dst.exists():
            dst = dest / f"{prefix}_{i}_{counter}_{src.name}"
            counter += 1
        shutil.copy2(src, dst)

# =========================================================
# LOAD IMAGES
# =========================================================
normal_images = get_images(NORMAL_SRC)
cysts_images = get_images(CYSTS_DIR)
stones_images = get_images(STONES_DIR)
tumors_images = get_images(TUMORS_DIR)

normal_count = len(normal_images)
cysts_count = len(cysts_images)
stones_count = len(stones_images)
tumors_count = len(tumors_images)

total_disease_count = cysts_count + stones_count + tumors_count

print("Source counts:")
print("Normal :", normal_count)
print("Cysts  :", cysts_count)
print("Stones :", stones_count)
print("Tumors :", tumors_count)
print("Total disease:", total_disease_count)

if normal_count == 0:
    raise ValueError("No normal images found.")

if total_disease_count == 0:
    raise ValueError("No disease images found.")

# =========================================================
# SAME PROPORTION TOTAL
# final normal count == final disease count
# =========================================================
target_count = min(normal_count, total_disease_count)

# =========================================================
# DISEASE SPLIT IN SAME PROPORTION
# =========================================================
raw_cysts = (cysts_count / total_disease_count) * target_count
raw_stones = (stones_count / total_disease_count) * target_count
raw_tumors = (tumors_count / total_disease_count) * target_count

pick_cysts = int(raw_cysts)
pick_stones = int(raw_stones)
pick_tumors = int(raw_tumors)

assigned = pick_cysts + pick_stones + pick_tumors
remaining = target_count - assigned

remainders = [
    ("cysts", raw_cysts - pick_cysts),
    ("stones", raw_stones - pick_stones),
    ("tumors", raw_tumors - pick_tumors),
]
remainders.sort(key=lambda x: x[1], reverse=True)

for cls_name, _ in remainders:
    if remaining <= 0:
        break
    if cls_name == "cysts" and pick_cysts < cysts_count:
        pick_cysts += 1
        remaining -= 1
    elif cls_name == "stones" and pick_stones < stones_count:
        pick_stones += 1
        remaining -= 1
    elif cls_name == "tumors" and pick_tumors < tumors_count:
        pick_tumors += 1
        remaining -= 1

print("\nSelected disease counts:")
print("Cysts picked :", pick_cysts)
print("Stones picked:", pick_stones)
print("Tumors picked:", pick_tumors)
print("Total disease picked:", pick_cysts + pick_stones + pick_tumors)

# =========================================================
# SAMPLE IMAGES
# =========================================================
selected_normal = random.sample(normal_images, target_count)

selected_cysts = random.sample(cysts_images, pick_cysts)
selected_stones = random.sample(stones_images, pick_stones)
selected_tumors = random.sample(tumors_images, pick_tumors)

selected_disease = (
    [("cysts", p) for p in selected_cysts] +
    [("stones", p) for p in selected_stones] +
    [("tumors", p) for p in selected_tumors]
)

# =========================================================
# CLEAN DESTINATION
# =========================================================
clean_folder(NORMAL_DEST)
clean_folder(DISEASE_DEST)

# =========================================================
# COPY NORMAL
# =========================================================
copy_images(selected_normal, NORMAL_DEST, prefix="normal")

# =========================================================
# COPY DISEASE
# =========================================================
for i, (cls_name, src) in enumerate(tqdm(selected_disease, desc="Copying disease"), start=1):
    dst = DISEASE_DEST / f"{cls_name}_{i}_{src.name}"
    counter = 1
    while dst.exists():
        dst = DISEASE_DEST / f"{cls_name}_{i}_{counter}_{src.name}"
        counter += 1
    shutil.copy2(src, dst)

# =========================================================
# FINAL SUMMARY
# =========================================================
print("\nDone.")
print("Saved normal to :", NORMAL_DEST)
print("Saved disease to:", DISEASE_DEST)
print("Final normal count :", len(list(NORMAL_DEST.iterdir())))
print("Final disease count:", len(list(DISEASE_DEST.iterdir())))

In [ ]:
from pathlib import Path

# =========================================================
# PATHS
# =========================================================
DISEASE_ROOT = Path(
    "/kaggle/input/datasets/ahmedshayan1/dataset-multicalasess/"
    "KidneyDiseaseClassification.folder/train"
)

CYSTS_DIR = DISEASE_ROOT / "cysts"
STONES_DIR = DISEASE_ROOT / "stones"
TUMORS_DIR = DISEASE_ROOT / "tumors"

NORMAL_DIR = Path(
    "/kaggle/input/datasets/gurjeetkaurmangat/"
    "kidney-ultrasound-images-stone-and-no-stone/"
    "my dataset final 512x512(implemented)/Normal"
)

STONE_DIR = Path(
    "/kaggle/input/datasets/gurjeetkaurmangat/"
    "kidney-ultrasound-images-stone-and-no-stone/"
    "my dataset final 512x512(implemented)/stone"
)

# =========================================================
# IMAGE EXTENSIONS
# =========================================================
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# =========================================================
# HELPER
# =========================================================
def count_images(folder: Path):
    if not folder.exists():
        return 0
    return sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

# =========================================================
# COUNTS
# =========================================================
cysts_count = count_images(CYSTS_DIR)
stones_count = count_images(STONES_DIR)
tumors_count = count_images(TUMORS_DIR)

normal_count = count_images(NORMAL_DIR)
stone_dataset_count = count_images(STONE_DIR)

total_disease_count = cysts_count + stones_count + tumors_count

# =========================================================
# PRINT
# =========================================================
print("Disease dataset counts:")
print(f"Cysts  : {cysts_count}")
print(f"Stones : {stones_count}")
print(f"Tumors : {tumors_count}")
print(f"Total disease (cysts + stones + tumors): {total_disease_count}")

print("\nStone/Normal dataset counts:")
print(f"Normal : {normal_count}")
print(f"Stone  : {stone_dataset_count}")

In [ ]:
from pathlib import Path

# =========================================================
# PATHS
# =========================================================
DISEASE_ROOT = Path(
    "/kaggle/input/datasets/ahmedshayan1/dataset-multicalasess/"
    "KidneyDiseaseClassification.folder/train"
)

CYSTS_DIR = DISEASE_ROOT / "cysts"
STONES_DIR = DISEASE_ROOT / "stones"
TUMORS_DIR = DISEASE_ROOT / "tumors"

NORMAL_DIR_1 = Path(
    "/kaggle/input/datasets/gurjeetkaurmangat/"
    "kidney-ultrasound-images-stone-and-no-stone/"
    "my dataset final 512x512(implemented)/Normal"
)

STONE_DIR_2 = Path(
    "/kaggle/input/datasets/gurjeetkaurmangat/"
    "kidney-ultrasound-images-stone-and-no-stone/"
    "my dataset final 512x512(implemented)/stone"
)

NORMAL_DIR_3 = Path("/kaggle/working/Kidney/normal")

# =========================================================
# IMAGE EXTENSIONS
# =========================================================
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# =========================================================
# FUNCTION TO COUNT IMAGES
# =========================================================
def count_images(folder: Path):
    if not folder.exists():
        return 0
    return sum(1 for f in folder.rglob("*") if f.is_file() and f.suffix.lower() in IMAGE_EXTS)

# =========================================================
# COUNTS
# =========================================================
cysts_count = count_images(CYSTS_DIR)
stones_count = count_images(STONES_DIR)
tumors_count = count_images(TUMORS_DIR)

total_diseased = cysts_count + stones_count + tumors_count

normal_1_count = count_images(NORMAL_DIR_1)
stone_2_count = count_images(STONE_DIR_2)
normal_3_count = count_images(NORMAL_DIR_3)

# =========================================================
# PRINT RESULTS
# =========================================================
print("===== DISEASE DATASET COUNTS =====")
print(f"Cysts images  : {cysts_count}")
print(f"Stones images : {stones_count}")
print(f"Tumors images : {tumors_count}")
print(f"Total diseased: {total_diseased}")

print("\n===== OTHER DATASET COUNTS =====")
print(f"Normal images (dataset input) : {normal_1_count}")
print(f"Stone images  (dataset input) : {stone_2_count}")
print(f"Normal images (/kaggle/working/Kidney/normal) : {normal_3_count}")

In [ ]:
import random
import shutil
from pathlib import Path
from tqdm import tqdm

# =========================================================
# SUGGESTED TARGET
# =========================================================
TARGET_PER_CLASS = 5000
SEED = 42
random.seed(SEED)

# =========================================================
# OUTPUT
# =========================================================
DEST_ROOT = Path("/kaggle/working/KidneyBalancedDataset")
NORMAL_DEST = DEST_ROOT / "normal"
DEFECTED_DEST = DEST_ROOT / "defected"

# =========================================================
# SOURCE PATHS
# =========================================================
NORMAL_DIR_1 = Path(
    "/kaggle/input/datasets/gurjeetkaurmangat/"
    "kidney-ultrasound-images-stone-and-no-stone/"
    "my dataset final 512x512(implemented)/Normal"
)

NORMAL_DIR_2 = Path("/kaggle/working/Kidney/normal")

DISEASE_ROOT = Path(
    "/kaggle/input/datasets/ahmedshayan1/dataset-multicalasess/"
    "KidneyDiseaseClassification.folder/train"
)

CYSTS_DIR = DISEASE_ROOT / "cysts"
STONES_DIR = DISEASE_ROOT / "stones"
TUMORS_DIR = DISEASE_ROOT / "tumors"

STONE_DIR_2 = Path(
    "/kaggle/input/datasets/gurjeetkaurmangat/"
    "kidney-ultrasound-images-stone-and-no-stone/"
    "my dataset final 512x512(implemented)/stone"
)

# =========================================================
# IMAGE EXTENSIONS
# =========================================================
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# =========================================================
# HELPERS
# =========================================================
def get_images(folder: Path):
    if not folder.exists():
        return []
    return [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS]

def clean_folder(folder: Path):
    folder.mkdir(parents=True, exist_ok=True)
    for f in folder.iterdir():
        if f.is_file():
            f.unlink()

def proportional_counts(source_counts: dict, target_total: int):
    total_available = sum(source_counts.values())
    raw = {name: (count / total_available) * target_total for name, count in source_counts.items()}
    picked = {name: int(v) for name, v in raw.items()}

    assigned = sum(picked.values())
    remaining = target_total - assigned

    remainders = sorted(
        [(name, raw[name] - picked[name]) for name in picked],
        key=lambda x: x[1],
        reverse=True
    )

    idx = 0
    while remaining > 0:
        name = remainders[idx % len(remainders)][0]
        if picked[name] < source_counts[name]:
            picked[name] += 1
            remaining -= 1
        idx += 1

    return picked

def copy_files(file_tuples, dest_dir: Path, prefix: str):
    dest_dir.mkdir(parents=True, exist_ok=True)
    copied = 0

    for i, (src_name, src_path) in enumerate(tqdm(file_tuples, desc=f"Copying -> {dest_dir.name}"), start=1):
        dst_name = f"{prefix}_{src_name}_{i}_{src_path.name}"
        dst_path = dest_dir / dst_name

        counter = 1
        while dst_path.exists():
            dst_path = dest_dir / f"{prefix}_{src_name}_{i}_{counter}_{src_path.name}"
            counter += 1

        shutil.copy2(src_path, dst_path)
        copied += 1

    return copied

# =========================================================
# LOAD IMAGES
# =========================================================
normal_sources = {
    "normal_dataset": get_images(NORMAL_DIR_1),
    "normal_working": get_images(NORMAL_DIR_2),
}

defected_sources = {
    "cysts": get_images(CYSTS_DIR),
    "stones_multi": get_images(STONES_DIR),
    "tumors": get_images(TUMORS_DIR),
    "stones_dataset": get_images(STONE_DIR_2),
}

# =========================================================
# COUNTS
# =========================================================
normal_counts = {name: len(imgs) for name, imgs in normal_sources.items()}
defected_counts = {name: len(imgs) for name, imgs in defected_sources.items()}

normal_total = sum(normal_counts.values())
defected_total = sum(defected_counts.values())

print("===== SOURCE COUNTS =====")
for name, count in normal_counts.items():
    print(f"{name:15s}: {count}")
for name, count in defected_counts.items():
    print(f"{name:15s}: {count}")

print(f"\nTotal normal available   : {normal_total}")
print(f"Total defected available : {defected_total}")

final_target = min(TARGET_PER_CLASS, normal_total, defected_total)
print(f"\nSuggested target per class: {TARGET_PER_CLASS}")
print(f"Final target per class    : {final_target}")

# =========================================================
# PRINT PROPORTIONS
# =========================================================
print("\n===== NORMAL PROPORTIONS =====")
for name, count in normal_counts.items():
    pct = (count / normal_total) * 100
    print(f"{name:15s}: {count:4d} ({pct:.2f}%)")

print("\n===== DEFECTED PROPORTIONS =====")
for name, count in defected_counts.items():
    pct = (count / defected_total) * 100
    print(f"{name:15s}: {count:4d} ({pct:.2f}%)")

# =========================================================
# PICK COUNTS
# =========================================================
normal_pick_counts = proportional_counts(normal_counts, final_target)
defected_pick_counts = proportional_counts(defected_counts, final_target)

print("\n===== NORMAL PICK COUNTS =====")
for name, count in normal_pick_counts.items():
    print(f"{name:15s}: {count}")

print("\n===== DEFECTED PICK COUNTS =====")
for name, count in defected_pick_counts.items():
    print(f"{name:15s}: {count}")

# =========================================================
# SAMPLE FILES
# =========================================================
selected_normal = []
for name, imgs in normal_sources.items():
    chosen = random.sample(imgs, normal_pick_counts[name])
    selected_normal.extend([(name, p) for p in chosen])

selected_defected = []
for name, imgs in defected_sources.items():
    chosen = random.sample(imgs, defected_pick_counts[name])
    selected_defected.extend([(name, p) for p in chosen])

# =========================================================
# PREPARE DESTINATION
# =========================================================
clean_folder(NORMAL_DEST)
clean_folder(DEFECTED_DEST)

# =========================================================
# COPY
# =========================================================
copied_normal = copy_files(selected_normal, NORMAL_DEST, "normal")
copied_defected = copy_files(selected_defected, DEFECTED_DEST, "defected")

# =========================================================
# FINAL STATS
# =========================================================
print("\n===== FINAL DATASET =====")
print(f"Saved normal   : {copied_normal}")
print(f"Saved defected : {copied_defected}")
print(f"Dataset path   : {DEST_ROOT}")

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install -U --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [2]:
import os
import copy
import csv
import random
from pathlib import Path

import numpy as np
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    balanced_accuracy_score,
    classification_report
)

ImageFile.LOAD_TRUNCATED_IMAGES = True

# =========================================================
# CONFIG
# =========================================================
TRAIN_DATASET_DIR = "/kaggle/working/KidneyBalancedDataset"
UNSEEN_TEST_DIR = "/kaggle/input/datasets/ahmedshayan1/test-unseen/Test"
OUTPUT_DIR = "/kaggle/vit_b16_effb3_ultrasound_benchmark"

IMG_SIZE = 224
BATCH_SIZE = 8            # lowered for ensemble memory usage
NUM_WORKERS = 0
RANDOM_SEED = 42

EPOCHS_STAGE1 = 5
EPOCHS_STAGE2 = 20
LR_STAGE1 = 1e-3
LR_STAGE2 = 1e-4
WEIGHT_DECAY = 1e-4
VAL_SPLIT = 0.2
EARLY_STOPPING_PATIENCE = 6

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# test folder name -> train folder name
TEST_TO_TRAIN_LABEL_MAP = {
    "Diseased": "defected",
    "Normal": "normal",
}

# =========================================================
# SEED
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)

# =========================================================
# FOCAL LOSS
# =========================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction="none")

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss
        return loss.mean()

# =========================================================
# DATASET
# =========================================================
class CustomImageDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("L")
        if self.transform:
            image = self.transform(image)
        return image, label, str(path)

# =========================================================
# TRANSFORMS
# =========================================================
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.80, 1.0), ratio=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.06, 0.06),
        scale=(0.90, 1.10)
    ),
    transforms.ColorJitter(brightness=0.20, contrast=0.20),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# =========================================================
# METRICS
# =========================================================
def calculate_metrics(y_true, y_pred, class_names):
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred)

    return {
        "acc": acc,
        "bal_acc": bal_acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "precision_per_class": precision_per_class,
        "recall_per_class": recall_per_class,
        "f1_per_class": f1_per_class,
        "support_per_class": support_per_class,
        "cm": cm,
        "report": classification_report(
            y_true,
            y_pred,
            target_names=class_names,
            digits=4,
            zero_division=0
        )
    }

def print_metrics(title, metrics, class_names):
    print(f"\n{'='*80}")
    print(title)
    print(f"{'='*80}")
    print(f"Accuracy           : {metrics['acc']:.4f}")
    print(f"Balanced Accuracy  : {metrics['bal_acc']:.4f}")
    print(f"Macro Precision    : {metrics['precision_macro']:.4f}")
    print(f"Macro Recall       : {metrics['recall_macro']:.4f}")
    print(f"Macro F1           : {metrics['f1_macro']:.4f}")
    print(f"Weighted F1        : {metrics['f1_weighted']:.4f}")

    print("\nPer-class Recall:")
    for i, cls in enumerate(class_names):
        print(f"  {cls}: {metrics['recall_per_class'][i]:.4f}")

    print("\nConfusion Matrix:")
    print(metrics["cm"])

    print("\nClassification Report:")
    print(metrics["report"])

# =========================================================
# MODEL - ViT-B/16 + EfficientNet-B3 Ensemble
# =========================================================
class ViTEfficientB3Ensemble(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        vit_weights = models.ViT_B_16_Weights.DEFAULT
        eff_weights = models.EfficientNet_B3_Weights.DEFAULT

        self.vit = models.vit_b_16(weights=vit_weights)
        self.eff = models.efficientnet_b3(weights=eff_weights)

        vit_in = self.vit.heads.head.in_features
        eff_in = self.eff.classifier[1].in_features

        self.vit.heads.head = nn.Identity()
        self.eff.classifier[1] = nn.Identity()

        self.vit_proj = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(vit_in, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.eff_proj = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(eff_in, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        vit_feat = self.vit(x)
        eff_feat = self.eff(x)

        vit_feat = self.vit_proj(vit_feat)
        eff_feat = self.eff_proj(eff_feat)

        fused = torch.cat([vit_feat, eff_feat], dim=1)
        logits = self.classifier(fused)
        return logits

def build_model(num_classes):
    return ViTEfficientB3Ensemble(num_classes=num_classes)

def freeze_backbones(model):
    for param in model.vit.parameters():
        param.requires_grad = False
    for param in model.eff.features.parameters():
        param.requires_grad = False

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

# =========================================================
# SAVE MISCLASSIFIED CSV
# =========================================================
def save_misclassified_csv(save_path, paths, y_true, y_pred, probs, class_names):
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "filepath",
            "true_label",
            "pred_label",
            "confidence_pred",
            *[f"prob_{cls}" for cls in class_names]
        ])

        for path, t, p, prob in zip(paths, y_true, y_pred, probs):
            if t != p:
                writer.writerow([
                    path,
                    class_names[t],
                    class_names[p],
                    float(prob[p]),
                    *[float(x) for x in prob]
                ])

# =========================================================
# EVALUATE
# =========================================================
def evaluate_model(model, loader, criterion, class_names, save_csv_path=None):
    model.eval()
    running_loss = 0.0

    all_labels = []
    all_preds = []
    all_probs = []
    all_paths = []

    with torch.no_grad():
        for images, labels, paths in tqdm(loader, desc="Evaluating", leave=False):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            running_loss += loss.item() * images.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_paths.extend(paths)

    avg_loss = running_loss / len(loader.dataset)
    metrics = calculate_metrics(all_labels, all_preds, class_names)
    metrics["loss"] = avg_loss

    if save_csv_path is not None:
        save_misclassified_csv(save_csv_path, all_paths, all_labels, all_preds, all_probs, class_names)

    return metrics

# =========================================================
# TRAIN ONE STAGE
# =========================================================
def train_stage(
    model,
    train_loader,
    val_loader,
    test_loader,
    optimizer,
    scheduler,
    criterion,
    class_names,
    epochs,
    history,
    best_val_macro_f1,
    save_name,
    stage_name
):
    best_model_wts = copy.deepcopy(model.state_dict())
    no_improve_epochs = 0

    for epoch in range(epochs):
        print(f"\n{stage_name} | Epoch [{epoch+1}/{epochs}]")
        print("-" * 80)

        model.train()
        running_train_loss = 0.0

        train_bar = tqdm(train_loader, desc=f"{stage_name} Train {epoch+1}", leave=False)
        for images, labels, _ in train_bar:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * images.size(0)
            train_bar.set_postfix(loss=f"{loss.item():.4f}")

        train_loss = running_train_loss / len(train_loader.dataset)

        val_metrics = evaluate_model(
            model=model,
            loader=val_loader,
            criterion=criterion,
            class_names=class_names
        )

        test_metrics = evaluate_model(
            model=model,
            loader=test_loader,
            criterion=criterion,
            class_names=class_names,
            save_csv_path=os.path.join(
                OUTPUT_DIR,
                f"{stage_name.lower().replace(' ', '_')}_epoch_{epoch+1}_test_misclassified.csv"
            )
        )

        if scheduler is not None:
            scheduler.step(val_metrics["f1_macro"])

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["val_macro_f1"].append(val_metrics["f1_macro"])
        history["val_bal_acc"].append(val_metrics["bal_acc"])
        history["test_loss"].append(test_metrics["loss"])
        history["test_macro_f1"].append(test_metrics["f1_macro"])
        history["test_bal_acc"].append(test_metrics["bal_acc"])

        print(f"Train Loss         : {train_loss:.4f}")

        print("\nValidation:")
        print(f"  Loss             : {val_metrics['loss']:.4f}")
        print(f"  Accuracy         : {val_metrics['acc']:.4f}")
        print(f"  Balanced Acc     : {val_metrics['bal_acc']:.4f}")
        print(f"  Macro F1         : {val_metrics['f1_macro']:.4f}")

        print("\nUnseen Benchmark Test:")
        print(f"  Loss             : {test_metrics['loss']:.4f}")
        print(f"  Accuracy         : {test_metrics['acc']:.4f}")
        print(f"  Balanced Acc     : {test_metrics['bal_acc']:.4f}")
        print(f"  Macro F1         : {test_metrics['f1_macro']:.4f}")

        print("\nValidation Confusion Matrix:")
        print(val_metrics["cm"])

        print("\nUnseen Test Confusion Matrix:")
        print(test_metrics["cm"])

        if val_metrics["f1_macro"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["f1_macro"]
            best_model_wts = copy.deepcopy(model.state_dict())
            no_improve_epochs = 0

            save_path = os.path.join(OUTPUT_DIR, save_name)
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_macro_f1": best_val_macro_f1,
                "class_names": class_names,
                "img_size": IMG_SIZE,
            }, save_path)
            print(f"\nSaved best model to: {save_path}")
        else:
            no_improve_epochs += 1
            print(f"\nNo validation improvement for {no_improve_epochs} epoch(s).")

        if no_improve_epochs >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_model_wts)
    return model, best_val_macro_f1, history

# =========================================================
# BUILD REMAPPED UNSEEN TEST SAMPLES
# =========================================================
def build_remapped_test_samples(unseen_test_dir, train_class_to_idx, mapping):
    unseen_test_dir = Path(unseen_test_dir)
    samples = []

    for test_folder_name, train_folder_name in mapping.items():
        folder = unseen_test_dir / test_folder_name
        if not folder.exists():
            raise FileNotFoundError(f"Missing test folder: {folder}")

        if train_folder_name not in train_class_to_idx:
            raise KeyError(f"Mapped train label '{train_folder_name}' not found in training classes")

        label_idx = train_class_to_idx[train_folder_name]

        image_paths = []
        for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff", "*.webp"):
            image_paths.extend(folder.glob(ext))
            image_paths.extend(folder.glob(ext.upper()))

        for img_path in sorted(image_paths):
            samples.append((str(img_path), label_idx))

    return samples

# =========================================================
# MAIN
# =========================================================
def main():
    train_base = datasets.ImageFolder(TRAIN_DATASET_DIR)

    print("TRAIN classes:", train_base.classes)
    print("TRAIN class_to_idx:", train_base.class_to_idx)
    print("TRAIN total images:", len(train_base.samples))

    class_names = train_base.classes
    train_class_to_idx = train_base.class_to_idx

    remapped_test_samples = build_remapped_test_samples(
        unseen_test_dir=UNSEEN_TEST_DIR,
        train_class_to_idx=train_class_to_idx,
        mapping=TEST_TO_TRAIN_LABEL_MAP
    )

    print("\nManual test label remap:")
    for k, v in TEST_TO_TRAIN_LABEL_MAP.items():
        print(f"  {k} -> {v}")

    print("UNSEEN TEST total images:", len(remapped_test_samples))

    all_samples = train_base.samples
    all_paths = [x[0] for x in all_samples]
    all_labels = [x[1] for x in all_samples]

    train_paths, val_paths, train_labels, val_labels = train_test_split(
        all_paths,
        all_labels,
        test_size=VAL_SPLIT,
        random_state=RANDOM_SEED,
        stratify=all_labels
    )

    train_samples = list(zip(train_paths, train_labels))
    val_samples = list(zip(val_paths, val_labels))
    test_samples = remapped_test_samples

    train_dataset = CustomImageDataset(train_samples, transform=train_transform)
    val_dataset = CustomImageDataset(val_samples, transform=eval_transform)
    test_dataset = CustomImageDataset(test_samples, transform=eval_transform)

    print("\nSplit summary:")
    print("Train size :", len(train_dataset))
    print("Val size   :", len(val_dataset))
    print("Test size  :", len(test_dataset))

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    model = build_model(num_classes=len(class_names)).to(DEVICE)
    criterion = FocalLoss(gamma=2.0)

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_macro_f1": [],
        "val_bal_acc": [],
        "test_loss": [],
        "test_macro_f1": [],
        "test_bal_acc": [],
    }

    best_val_macro_f1 = 0.0

    print("\n" + "=" * 80)
    print("STAGE 1: Train fusion head only")
    print("=" * 80)

    freeze_backbones(model)

    optimizer_stage1 = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_STAGE1,
        weight_decay=WEIGHT_DECAY
    )

    scheduler_stage1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_stage1,
        mode="max",
        factor=0.5,
        patience=2
    )

    model, best_val_macro_f1, history = train_stage(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        optimizer=optimizer_stage1,
        scheduler=scheduler_stage1,
        criterion=criterion,
        class_names=class_names,
        epochs=EPOCHS_STAGE1,
        history=history,
        best_val_macro_f1=best_val_macro_f1,
        save_name="best_stage1_vit_effb3_ensemble.pth",
        stage_name="Stage 1"
    )

    print("\n" + "=" * 80)
    print("STAGE 2: Fine-tune full ensemble")
    print("=" * 80)

    unfreeze_all(model)

    optimizer_stage2 = torch.optim.AdamW(
        model.parameters(),
        lr=LR_STAGE2,
        weight_decay=WEIGHT_DECAY
    )

    scheduler_stage2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_stage2,
        mode="max",
        factor=0.5,
        patience=2
    )

    model, best_val_macro_f1, history = train_stage(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        optimizer=optimizer_stage2,
        scheduler=scheduler_stage2,
        criterion=criterion,
        class_names=class_names,
        epochs=EPOCHS_STAGE2,
        history=history,
        best_val_macro_f1=best_val_macro_f1,
        save_name="best_stage2_vit_effb3_ensemble.pth",
        stage_name="Stage 2"
    )

    final_model_path = os.path.join(OUTPUT_DIR, "final_vit_effb3_ensemble_ultrasound.pth")
    torch.save({
        "model_state_dict": model.state_dict(),
        "class_names": class_names,
        "img_size": IMG_SIZE,
        "best_val_macro_f1": best_val_macro_f1,
    }, final_model_path)

    print(f"\nFinal model saved to: {final_model_path}")
    print(f"Best Validation Macro F1: {best_val_macro_f1:.4f}")

    final_val_metrics = evaluate_model(
        model=model,
        loader=val_loader,
        criterion=criterion,
        class_names=class_names,
        save_csv_path=os.path.join(OUTPUT_DIR, "final_val_misclassified.csv")
    )
    print_metrics("FINAL VALIDATION RESULTS", final_val_metrics, class_names)

    final_test_metrics = evaluate_model(
        model=model,
        loader=test_loader,
        criterion=criterion,
        class_names=class_names,
        save_csv_path=os.path.join(OUTPUT_DIR, "final_unseen_test_misclassified.csv")
    )
    print_metrics("FINAL UNSEEN BENCHMARK RESULTS", final_test_metrics, class_names)

if __name__ == "__main__":
    main()

TRAIN classes: ['defected', 'normal']
TRAIN class_to_idx: {'defected': 0, 'normal': 1}
TRAIN total images: 10000

Manual test label remap:
  Diseased -> defected
  Normal -> normal
UNSEEN TEST total images: 20

Split summary:
Train size : 8000
Val size   : 2000
Test size  : 20

STAGE 1: Train fusion head only

Stage 1 | Epoch [1/5]
--------------------------------------------------------------------------------


Stage 1 Train 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/250 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Train Loss         : 0.1042

Validation:
  Loss             : 0.0964
  Accuracy         : 0.8400
  Balanced Acc     : 0.8400
  Macro F1         : 0.8395

Unseen Benchmark Test:
  Loss             : 0.2127
  Accuracy         : 0.5000
  Balanced Acc     : 0.5000
  Macro F1         : 0.4949

Validation Confusion Matrix:
[[785 215]
 [105 895]]

Unseen Test Confusion Matrix:
[[6 6]
 [4 4]]

Saved best model to: /kaggle/vit_b16_effb3_ultrasound_benchmark/best_stage1_vit_effb3_ensemble.pth

Stage 1 | Epoch [2/5]
--------------------------------------------------------------------------------


Stage 1 Train 2:   0%|          | 0/1000 [00:00<?, ?it/s]

... (33 outputs left out)

Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Train Loss         : 0.0593

Validation:
  Loss             : 0.0562
  Accuracy         : 0.8785
  Balanced Acc     : 0.8785
  Macro F1         : 0.8775

Unseen Benchmark Test:
  Loss             : 0.4339
  Accuracy         : 0.5500
  Balanced Acc     : 0.5208
  Macro F1         : 0.5200

Validation Confusion Matrix:
[[787 213]
 [ 30 970]]

Unseen Test Confusion Matrix:
[[8 4]
 [5 3]]

No validation improvement for 4 epoch(s).

Stage 2 | Epoch [6/20]
--------------------------------------------------------------------------------


Stage 2 Train 6:   0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import os
import copy
import csv
import random
from pathlib import Path

import numpy as np
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    balanced_accuracy_score,
    classification_report,
)

ImageFile.LOAD_TRUNCATED_IMAGES = True

# =========================================================
# CONFIG
# =========================================================
TRAIN_DATASET_DIR = "/kaggle/working/KidneyBalancedDataset"
UNSEEN_TEST_DIR = "/kaggle/input/datasets/ahmedshayan1/test-unseen/Test"
OUTPUT_DIR = "/kaggle/working/effb3_ultrasound_generalization"

IMG_SIZE = 300
BATCH_SIZE = 16
NUM_WORKERS = 2
RANDOM_SEED = 42

EPOCHS_STAGE1 = 5     # classifier only
EPOCHS_STAGE2 = 8     # unfreeze last block
EPOCHS_STAGE3 = 12    # full fine-tune

LR_STAGE1 = 1e-3
LR_STAGE2 = 3e-4
LR_STAGE3 = 1e-4
WEIGHT_DECAY = 1e-4
VAL_SPLIT = 0.2
EARLY_STOPPING_PATIENCE = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# test folder name -> train folder name
TEST_TO_TRAIN_LABEL_MAP = {
    "Diseased": "defected",
    "Normal": "normal",
}

# =========================================================
# SEED
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)

# =========================================================
# DATASET
# =========================================================
class CustomImageDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("L")
        if self.transform:
            image = self.transform(image)
        return image, label, str(path)

# =========================================================
# MILDER TRANSFORMS
# =========================================================
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20)),
    transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# =========================================================
# METRICS
# =========================================================
def calculate_metrics(y_true, y_pred, class_names):
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred)

    return {
        "acc": acc,
        "bal_acc": bal_acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "precision_per_class": precision_per_class,
        "recall_per_class": recall_per_class,
        "f1_per_class": f1_per_class,
        "support_per_class": support_per_class,
        "cm": cm,
        "report": classification_report(
            y_true,
            y_pred,
            target_names=class_names,
            digits=4,
            zero_division=0
        )
    }

def print_metrics(title, metrics, class_names):
    print(f"\n{'='*80}")
    print(title)
    print(f"{'='*80}")
    print(f"Accuracy           : {metrics['acc']:.4f}")
    print(f"Balanced Accuracy  : {metrics['bal_acc']:.4f}")
    print(f"Macro Precision    : {metrics['precision_macro']:.4f}")
    print(f"Macro Recall       : {metrics['recall_macro']:.4f}")
    print(f"Macro F1           : {metrics['f1_macro']:.4f}")
    print(f"Weighted F1        : {metrics['f1_weighted']:.4f}")

    print("\nPer-class Recall:")
    for i, cls in enumerate(class_names):
        print(f"  {cls}: {metrics['recall_per_class'][i]:.4f}")

    print("\nConfusion Matrix:")
    print(metrics["cm"])

    print("\nClassification Report:")
    print(metrics["report"])

# =========================================================
# MODEL - SINGLE EFFICIENTNET-B3
# =========================================================
class EfficientNetB3Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        weights = models.EfficientNet_B3_Weights.DEFAULT
        self.backbone = models.efficientnet_b3(weights=weights)
        in_features = self.backbone.classifier[1].in_features

        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.35),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.25),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

def build_model(num_classes):
    return EfficientNetB3Classifier(num_classes=num_classes)

# =========================================================
# FREEZING / UNFREEZING
# =========================================================
def freeze_all_features(model):
    for param in model.backbone.features.parameters():
        param.requires_grad = False

def unfreeze_last_n_feature_blocks(model, n=2):
    blocks = list(model.backbone.features.children())
    for block in blocks[-n:]:
        for param in block.parameters():
            param.requires_grad = True

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

def print_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# =========================================================
# SAVE MISCLASSIFIED CSV
# =========================================================
def save_misclassified_csv(save_path, paths, y_true, y_pred, probs, class_names):
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "filepath",
            "true_label",
            "pred_label",
            "confidence_pred",
            *[f"prob_{cls}" for cls in class_names]
        ])

        for path, t, p, prob in zip(paths, y_true, y_pred, probs):
            if t != p:
                writer.writerow([
                    path,
                    class_names[t],
                    class_names[p],
                    float(prob[p]),
                    *[float(x) for x in prob]
                ])

# =========================================================
# EVALUATE
# =========================================================
def evaluate_model(model, loader, criterion, class_names, save_csv_path=None):
    model.eval()
    running_loss = 0.0

    all_labels = []
    all_preds = []
    all_probs = []
    all_paths = []

    with torch.no_grad():
        for images, labels, paths in tqdm(loader, desc="Evaluating", leave=False):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            running_loss += loss.item() * images.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_paths.extend(paths)

    avg_loss = running_loss / len(loader.dataset)
    metrics = calculate_metrics(all_labels, all_preds, class_names)
    metrics["loss"] = avg_loss

    if save_csv_path is not None:
        save_misclassified_csv(save_csv_path, all_paths, all_labels, all_preds, all_probs, class_names)

    return metrics

# =========================================================
# TRAIN STAGE
# =========================================================
def train_stage(
    model,
    train_loader,
    val_loader,
    test_loader,
    optimizer,
    scheduler,
    criterion,
    class_names,
    epochs,
    history,
    best_val_macro_f1,
    save_name,
    stage_name,
):
    best_model_wts = copy.deepcopy(model.state_dict())
    no_improve_epochs = 0

    for epoch in range(epochs):
        print(f"\n{stage_name} | Epoch [{epoch+1}/{epochs}]")
        print("-" * 80)

        model.train()
        running_train_loss = 0.0

        train_bar = tqdm(train_loader, desc=f"{stage_name} Train {epoch+1}", leave=False)
        for images, labels, _ in train_bar:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * images.size(0)
            train_bar.set_postfix(loss=f"{loss.item():.4f}")

        train_loss = running_train_loss / len(train_loader.dataset)

        val_metrics = evaluate_model(
            model=model,
            loader=val_loader,
            criterion=criterion,
            class_names=class_names
        )

        test_metrics = evaluate_model(
            model=model,
            loader=test_loader,
            criterion=criterion,
            class_names=class_names,
            save_csv_path=os.path.join(
                OUTPUT_DIR,
                f"{stage_name.lower().replace(' ', '_')}_epoch_{epoch+1}_test_misclassified.csv"
            )
        )

        if scheduler is not None:
            scheduler.step(val_metrics["f1_macro"])

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["val_macro_f1"].append(val_metrics["f1_macro"])
        history["val_bal_acc"].append(val_metrics["bal_acc"])
        history["test_loss"].append(test_metrics["loss"])
        history["test_macro_f1"].append(test_metrics["f1_macro"])
        history["test_bal_acc"].append(test_metrics["bal_acc"])

        print(f"Train Loss         : {train_loss:.4f}")

        print("\nValidation:")
        print(f"  Loss             : {val_metrics['loss']:.4f}")
        print(f"  Accuracy         : {val_metrics['acc']:.4f}")
        print(f"  Balanced Acc     : {val_metrics['bal_acc']:.4f}")
        print(f"  Macro F1         : {val_metrics['f1_macro']:.4f}")

        print("\nUnseen Benchmark Test:")
        print(f"  Loss             : {test_metrics['loss']:.4f}")
        print(f"  Accuracy         : {test_metrics['acc']:.4f}")
        print(f"  Balanced Acc     : {test_metrics['bal_acc']:.4f}")
        print(f"  Macro F1         : {test_metrics['f1_macro']:.4f}")

        print("\nValidation Confusion Matrix:")
        print(val_metrics["cm"])

        print("\nUnseen Test Confusion Matrix:")
        print(test_metrics["cm"])

        if val_metrics["f1_macro"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["f1_macro"]
            best_model_wts = copy.deepcopy(model.state_dict())
            no_improve_epochs = 0

            save_path = os.path.join(OUTPUT_DIR, save_name)
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_macro_f1": best_val_macro_f1,
                "class_names": class_names,
                "img_size": IMG_SIZE,
            }, save_path)
            print(f"\nSaved best model to: {save_path}")
        else:
            no_improve_epochs += 1
            print(f"\nNo validation improvement for {no_improve_epochs} epoch(s).")

        if no_improve_epochs >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_model_wts)
    return model, best_val_macro_f1, history

# =========================================================
# BUILD REMAPPED UNSEEN TEST SAMPLES
# =========================================================
def build_remapped_test_samples(unseen_test_dir, train_class_to_idx, mapping):
    unseen_test_dir = Path(unseen_test_dir)
    samples = []

    for test_folder_name, train_folder_name in mapping.items():
        folder = unseen_test_dir / test_folder_name
        if not folder.exists():
            raise FileNotFoundError(f"Missing test folder: {folder}")

        if train_folder_name not in train_class_to_idx:
            raise KeyError(f"Mapped train label '{train_folder_name}' not found in training classes")

        label_idx = train_class_to_idx[train_folder_name]

        image_paths = []
        for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff", "*.webp"):
            image_paths.extend(folder.glob(ext))
            image_paths.extend(folder.glob(ext.upper()))

        for img_path in sorted(image_paths):
            samples.append((str(img_path), label_idx))

    return samples

# =========================================================
# MAIN
# =========================================================
def main():
    train_base = datasets.ImageFolder(TRAIN_DATASET_DIR)

    print("TRAIN classes:", train_base.classes)
    print("TRAIN class_to_idx:", train_base.class_to_idx)
    print("TRAIN total images:", len(train_base.samples))

    class_names = train_base.classes
    train_class_to_idx = train_base.class_to_idx

    remapped_test_samples = build_remapped_test_samples(
        unseen_test_dir=UNSEEN_TEST_DIR,
        train_class_to_idx=train_class_to_idx,
        mapping=TEST_TO_TRAIN_LABEL_MAP
    )

    print("\nManual test label remap:")
    for k, v in TEST_TO_TRAIN_LABEL_MAP.items():
        print(f"  {k} -> {v}")

    print("UNSEEN TEST total images:", len(remapped_test_samples))

    all_samples = train_base.samples
    all_paths = [x[0] for x in all_samples]
    all_labels = [x[1] for x in all_samples]

    train_paths, val_paths, train_labels, val_labels = train_test_split(
        all_paths,
        all_labels,
        test_size=VAL_SPLIT,
        random_state=RANDOM_SEED,
        stratify=all_labels
    )

    train_samples = list(zip(train_paths, train_labels))
    val_samples = list(zip(val_paths, val_labels))
    test_samples = remapped_test_samples

    train_dataset = CustomImageDataset(train_samples, transform=train_transform)
    val_dataset = CustomImageDataset(val_samples, transform=eval_transform)
    test_dataset = CustomImageDataset(test_samples, transform=eval_transform)

    print("\nSplit summary:")
    print("Train size :", len(train_dataset))
    print("Val size   :", len(val_dataset))
    print("Test size  :", len(test_dataset))

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    model = build_model(num_classes=len(class_names)).to(DEVICE)

    # label smoothing instead of focal loss
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_macro_f1": [],
        "val_bal_acc": [],
        "test_loss": [],
        "test_macro_f1": [],
        "test_bal_acc": [],
    }

    best_val_macro_f1 = 0.0

    # =====================================================
    # STAGE 1 - classifier only
    # =====================================================
    print("\n" + "=" * 80)
    print("STAGE 1: Train classifier only")
    print("=" * 80)

    freeze_all_features(model)
    print_trainable_params(model)

    optimizer_stage1 = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_STAGE1,
        weight_decay=WEIGHT_DECAY
    )

    scheduler_stage1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_stage1,
        mode="max",
        factor=0.5,
        patience=2
    )

    model, best_val_macro_f1, history = train_stage(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        optimizer=optimizer_stage1,
        scheduler=scheduler_stage1,
        criterion=criterion,
        class_names=class_names,
        epochs=EPOCHS_STAGE1,
        history=history,
        best_val_macro_f1=best_val_macro_f1,
        save_name="best_stage1_effb3.pth",
        stage_name="Stage 1"
    )

    # =====================================================
    # STAGE 2 - last blocks only
    # =====================================================
    print("\n" + "=" * 80)
    print("STAGE 2: Unfreeze last 2 feature blocks")
    print("=" * 80)

    unfreeze_last_n_feature_blocks(model, n=2)
    print_trainable_params(model)

    optimizer_stage2 = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_STAGE2,
        weight_decay=WEIGHT_DECAY
    )

    scheduler_stage2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_stage2,
        mode="max",
        factor=0.5,
        patience=2
    )

    model, best_val_macro_f1, history = train_stage(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        optimizer=optimizer_stage2,
        scheduler=scheduler_stage2,
        criterion=criterion,
        class_names=class_names,
        epochs=EPOCHS_STAGE2,
        history=history,
        best_val_macro_f1=best_val_macro_f1,
        save_name="best_stage2_effb3.pth",
        stage_name="Stage 2"
    )

    # =====================================================
    # STAGE 3 - full fine-tune
    # =====================================================
    print("\n" + "=" * 80)
    print("STAGE 3: Fine-tune full model")
    print("=" * 80)

    unfreeze_all(model)
    print_trainable_params(model)

    optimizer_stage3 = torch.optim.AdamW(
        model.parameters(),
        lr=LR_STAGE3,
        weight_decay=WEIGHT_DECAY
    )

    scheduler_stage3 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_stage3,
        mode="max",
        factor=0.5,
        patience=2
    )

    model, best_val_macro_f1, history = train_stage(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        optimizer=optimizer_stage3,
        scheduler=scheduler_stage3,
        criterion=criterion,
        class_names=class_names,
        epochs=EPOCHS_STAGE3,
        history=history,
        best_val_macro_f1=best_val_macro_f1,
        save_name="best_stage3_effb3.pth",
        stage_name="Stage 3"
    )

    final_model_path = os.path.join(OUTPUT_DIR, "final_effb3_ultrasound_classifier.pth")
    torch.save({
        "model_state_dict": model.state_dict(),
        "class_names": class_names,
        "img_size": IMG_SIZE,
        "best_val_macro_f1": best_val_macro_f1,
    }, final_model_path)

    print(f"\nFinal model saved to: {final_model_path}")
    print(f"Best Validation Macro F1: {best_val_macro_f1:.4f}")

    final_val_metrics = evaluate_model(
        model=model,
        loader=val_loader,
        criterion=criterion,
        class_names=class_names,
        save_csv_path=os.path.join(OUTPUT_DIR, "final_val_misclassified.csv")
    )
    print_metrics("FINAL VALIDATION RESULTS", final_val_metrics, class_names)

    final_test_metrics = evaluate_model(
        model=model,
        loader=test_loader,
        criterion=criterion,
        class_names=class_names,
        save_csv_path=os.path.join(OUTPUT_DIR, "final_unseen_test_misclassified.csv")
    )
    print_metrics("FINAL UNSEEN BENCHMARK RESULTS", final_test_metrics, class_names)

if __name__ == "__main__":
    main()

In [3]:
import os
import random
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split

ImageFile.LOAD_TRUNCATED_IMAGES = True

# =========================================================
# CONFIG
# =========================================================
TRAIN_DATASET_DIR = "/kaggle/working/KidneyBalancedDataset"
IMG_SIZE = 300
BATCH_SIZE = 16
NUM_WORKERS = 2
VAL_SPLIT = 0.2
RANDOM_SEED = 42

# =========================================================
# SEED
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)

# =========================================================
# PREPROCESSING FUNCTIONS
# =========================================================
def remove_black_border(gray_img, threshold=8):
    """
    Remove large black borders/background around ultrasound image.
    Expects uint8 grayscale image.
    """
    mask = gray_img > threshold
    coords = np.argwhere(mask)

    if coords.size == 0:
        return gray_img

    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1
    cropped = gray_img[y0:y1, x0:x1]

    return cropped

def apply_median_denoise(gray_img, ksize=3):
    """
    Mild speckle-noise reduction.
    """
    return cv2.medianBlur(gray_img, ksize)

def apply_clahe(gray_img, clip_limit=2.0, tile_grid_size=(8, 8)):
    """
    Contrast Limited Adaptive Histogram Equalization.
    """
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(gray_img)

def preprocess_ultrasound_gray(gray_img):
    """
    Full preprocessing pipeline:
    1. border removal
    2. mild denoise
    3. CLAHE
    """
    img = remove_black_border(gray_img, threshold=8)
    img = apply_median_denoise(img, ksize=3)
    img = apply_clahe(img, clip_limit=2.0, tile_grid_size=(8, 8))
    return img

# =========================================================
# CUSTOM DATASET
# =========================================================
class KidneyUltrasoundDataset(Dataset):
    def __init__(self, samples, transform=None, apply_preprocessing=True):
        self.samples = samples
        self.transform = transform
        self.apply_preprocessing = apply_preprocessing

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        # read as grayscale
        gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if gray is None:
            raise ValueError(f"Could not read image: {path}")

        if self.apply_preprocessing:
            gray = preprocess_ultrasound_gray(gray)

        # convert to PIL image for torchvision transforms
        image = Image.fromarray(gray)

        if self.transform:
            image = self.transform(image)

        return image, label, str(path)

# =========================================================
# TRANSFORMS
# =========================================================
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20)),
    transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# =========================================================
# BUILD TRAIN / VAL ONLY
# =========================================================
def build_train_val_loaders():
    base_dataset = datasets.ImageFolder(TRAIN_DATASET_DIR)

    print("TRAIN classes:", base_dataset.classes)
    print("TRAIN class_to_idx:", base_dataset.class_to_idx)
    print("TRAIN total images:", len(base_dataset.samples))

    all_samples = base_dataset.samples
    all_paths = [x[0] for x in all_samples]
    all_labels = [x[1] for x in all_samples]

    train_paths, val_paths, train_labels, val_labels = train_test_split(
        all_paths,
        all_labels,
        test_size=VAL_SPLIT,
        random_state=RANDOM_SEED,
        stratify=all_labels
    )

    train_samples = list(zip(train_paths, train_labels))
    val_samples = list(zip(val_paths, val_labels))

    train_dataset = KidneyUltrasoundDataset(
        samples=train_samples,
        transform=train_transform,
        apply_preprocessing=True
    )

    val_dataset = KidneyUltrasoundDataset(
        samples=val_samples,
        transform=val_transform,
        apply_preprocessing=True
    )

    print("\nSplit summary:")
    print("Train size :", len(train_dataset))
    print("Val size   :", len(val_dataset))

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    return train_loader, val_loader, base_dataset.classes

# =========================================================
# OPTIONAL: PREVIEW SOME PREPROCESSED SAMPLES
# =========================================================
def inspect_preprocessing(samples, num_samples=5):
    print("\nInspecting preprocessing samples...\n")
    for i, (path, label) in enumerate(samples[:num_samples], start=1):
        gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        proc = preprocess_ultrasound_gray(gray)

        print(f"Sample {i}")
        print(f"  Path            : {path}")
        print(f"  Original shape  : {gray.shape}")
        print(f"  Processed shape : {proc.shape}")
        print(f"  Label           : {label}")
        print("-" * 60)

# =========================================================
# MAIN TEST
# =========================================================
if __name__ == "__main__":
    train_loader, val_loader, class_names = build_train_val_loaders()

    # show one batch shape
    train_images, train_labels, train_paths = next(iter(train_loader))
    val_images, val_labels, val_paths = next(iter(val_loader))

    print("\nBatch check:")
    print("Train batch images shape:", train_images.shape)
    print("Train batch labels shape:", train_labels.shape)
    print("Val batch images shape  :", val_images.shape)
    print("Val batch labels shape  :", val_labels.shape)
    print("Classes:", class_names)

TRAIN classes: ['defected', 'normal']
TRAIN class_to_idx: {'defected': 0, 'normal': 1}
TRAIN total images: 10000

Split summary:
Train size : 8000
Val size   : 2000

Batch check:
Train batch images shape: torch.Size([16, 3, 300, 300])
Train batch labels shape: torch.Size([16])
Val batch images shape  : torch.Size([16, 3, 300, 300])
Val batch labels shape  : torch.Size([16])
Classes: ['defected', 'normal']


In [ ]:
import os
import copy
import csv
import random
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    balanced_accuracy_score,
    classification_report
)

ImageFile.LOAD_TRUNCATED_IMAGES = True

# =========================================================
# CONFIG
# =========================================================
TRAIN_DATASET_DIR = "/kaggle/working/KidneyBalancedDataset"
UNSEEN_TEST_DIR = "/kaggle/input/datasets/ahmedshayan1/test-unseen/Test"
OUTPUT_DIR = "/kaggle/working/effb3_kidney_ultrasound"

IMG_SIZE = 300
BATCH_SIZE = 16
NUM_WORKERS = 2
RANDOM_SEED = 42

EPOCHS_STAGE1 = 5
EPOCHS_STAGE2 = 8
EPOCHS_STAGE3 = 12

LR_STAGE1 = 1e-3
LR_STAGE2 = 3e-4
LR_STAGE3 = 1e-4
WEIGHT_DECAY = 1e-4
VAL_SPLIT = 0.2
EARLY_STOPPING_PATIENCE = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# test folder name -> train folder name
TEST_TO_TRAIN_LABEL_MAP = {
    "Diseased": "defected",
    "Normal": "normal",
}

# =========================================================
# SEED
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)

# =========================================================
# PREPROCESSING
# =========================================================
def remove_black_border(gray_img, threshold=8):
    mask = gray_img > threshold
    coords = np.argwhere(mask)

    if coords.size == 0:
        return gray_img

    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1
    return gray_img[y0:y1, x0:x1]

def apply_median_denoise(gray_img, ksize=3):
    return cv2.medianBlur(gray_img, ksize)

def apply_clahe(gray_img, clip_limit=2.0, tile_grid_size=(8, 8)):
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(gray_img)

def preprocess_ultrasound_gray(gray_img):
    img = remove_black_border(gray_img, threshold=8)
    img = apply_median_denoise(img, ksize=3)
    img = apply_clahe(img, clip_limit=2.0, tile_grid_size=(8, 8))
    return img

# =========================================================
# DATASET
# =========================================================
class KidneyUltrasoundDataset(Dataset):
    def __init__(self, samples, transform=None, apply_preprocessing=True):
        self.samples = samples
        self.transform = transform
        self.apply_preprocessing = apply_preprocessing

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if gray is None:
            raise ValueError(f"Could not read image: {path}")

        if self.apply_preprocessing:
            gray = preprocess_ultrasound_gray(gray)

        image = Image.fromarray(gray)

        if self.transform:
            image = self.transform(image)

        return image, label, str(path)

# =========================================================
# TRANSFORMS
# =========================================================
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20)),
    transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# =========================================================
# METRICS
# =========================================================
def calculate_metrics(y_true, y_pred, class_names):
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred)

    return {
        "acc": acc,
        "bal_acc": bal_acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "precision_per_class": precision_per_class,
        "recall_per_class": recall_per_class,
        "f1_per_class": f1_per_class,
        "support_per_class": support_per_class,
        "cm": cm,
        "report": classification_report(
            y_true,
            y_pred,
            target_names=class_names,
            digits=4,
            zero_division=0
        )
    }

def print_metrics(title, metrics, class_names):
    print(f"\n{'='*80}")
    print(title)
    print(f"{'='*80}")
    print(f"Accuracy           : {metrics['acc']:.4f}")
    print(f"Balanced Accuracy  : {metrics['bal_acc']:.4f}")
    print(f"Macro Precision    : {metrics['precision_macro']:.4f}")
    print(f"Macro Recall       : {metrics['recall_macro']:.4f}")
    print(f"Macro F1           : {metrics['f1_macro']:.4f}")
    print(f"Weighted F1        : {metrics['f1_weighted']:.4f}")

    print("\nPer-class Recall:")
    for i, cls in enumerate(class_names):
        print(f"  {cls}: {metrics['recall_per_class'][i]:.4f}")

    print("\nConfusion Matrix:")
    print(metrics["cm"])

    print("\nClassification Report:")
    print(metrics["report"])

# =========================================================
# MODEL
# =========================================================
class EfficientNetB3Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        weights = models.EfficientNet_B3_Weights.DEFAULT
        self.backbone = models.efficientnet_b3(weights=weights)
        in_features = self.backbone.classifier[1].in_features

        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.35),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.25),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

def build_model(num_classes):
    return EfficientNetB3Classifier(num_classes=num_classes)

# =========================================================
# FREEZE / UNFREEZE
# =========================================================
def freeze_all_features(model):
    for param in model.backbone.features.parameters():
        param.requires_grad = False

def unfreeze_last_n_feature_blocks(model, n=2):
    blocks = list(model.backbone.features.children())
    for block in blocks[-n:]:
        for param in block.parameters():
            param.requires_grad = True

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

def print_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

# =========================================================
# SAVE MISCLASSIFIED CSV
# =========================================================
def save_misclassified_csv(save_path, paths, y_true, y_pred, probs, class_names):
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "filepath",
            "true_label",
            "pred_label",
            "confidence_pred",
            *[f"prob_{cls}" for cls in class_names]
        ])

        for path, t, p, prob in zip(paths, y_true, y_pred, probs):
            if t != p:
                writer.writerow([
                    path,
                    class_names[t],
                    class_names[p],
                    float(prob[p]),
                    *[float(x) for x in prob]
                ])

# =========================================================
# EVALUATE
# =========================================================
def evaluate_model(model, loader, criterion, class_names, save_csv_path=None):
    model.eval()
    running_loss = 0.0

    all_labels = []
    all_preds = []
    all_probs = []
    all_paths = []

    with torch.no_grad():
        for images, labels, paths in tqdm(loader, desc="Evaluating", leave=False):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            running_loss += loss.item() * images.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_paths.extend(paths)

    avg_loss = running_loss / len(loader.dataset)
    metrics = calculate_metrics(all_labels, all_preds, class_names)
    metrics["loss"] = avg_loss

    if save_csv_path is not None:
        save_misclassified_csv(save_csv_path, all_paths, all_labels, all_preds, all_probs, class_names)

    return metrics

# =========================================================
# TRAIN STAGE
# =========================================================
def train_stage(
    model,
    train_loader,
    val_loader,
    unseen_loader,
    optimizer,
    scheduler,
    criterion,
    class_names,
    epochs,
    history,
    best_val_macro_f1,
    save_name,
    stage_name,
):
    best_model_wts = copy.deepcopy(model.state_dict())
    no_improve_epochs = 0

    for epoch in range(epochs):
        print(f"\n{stage_name} | Epoch [{epoch+1}/{epochs}]")
        print("-" * 80)

        model.train()
        running_train_loss = 0.0

        train_bar = tqdm(train_loader, desc=f"{stage_name} Train {epoch+1}", leave=False)
        for images, labels, _ in train_bar:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * images.size(0)
            train_bar.set_postfix(loss=f"{loss.item():.4f}")

        train_loss = running_train_loss / len(train_loader.dataset)

        val_metrics = evaluate_model(
            model=model,
            loader=val_loader,
            criterion=criterion,
            class_names=class_names
        )

        unseen_metrics = evaluate_model(
            model=model,
            loader=unseen_loader,
            criterion=criterion,
            class_names=class_names,
            save_csv_path=os.path.join(
                OUTPUT_DIR,
                f"{stage_name.lower().replace(' ', '_')}_epoch_{epoch+1}_unseen_misclassified.csv"
            )
        )

        if scheduler is not None:
            scheduler.step(val_metrics["f1_macro"])

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["val_macro_f1"].append(val_metrics["f1_macro"])
        history["val_bal_acc"].append(val_metrics["bal_acc"])
        history["unseen_loss"].append(unseen_metrics["loss"])
        history["unseen_macro_f1"].append(unseen_metrics["f1_macro"])
        history["unseen_bal_acc"].append(unseen_metrics["bal_acc"])

        print(f"Train Loss         : {train_loss:.4f}")

        print("\nValidation:")
        print(f"  Loss             : {val_metrics['loss']:.4f}")
        print(f"  Accuracy         : {val_metrics['acc']:.4f}")
        print(f"  Balanced Acc     : {val_metrics['bal_acc']:.4f}")
        print(f"  Macro F1         : {val_metrics['f1_macro']:.4f}")

        print("\nUnseen Benchmark Test:")
        print(f"  Loss             : {unseen_metrics['loss']:.4f}")
        print(f"  Accuracy         : {unseen_metrics['acc']:.4f}")
        print(f"  Balanced Acc     : {unseen_metrics['bal_acc']:.4f}")
        print(f"  Macro F1         : {unseen_metrics['f1_macro']:.4f}")

        print("\nValidation Confusion Matrix:")
        print(val_metrics["cm"])

        print("\nUnseen Test Confusion Matrix:")
        print(unseen_metrics["cm"])

        if val_metrics["f1_macro"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["f1_macro"]
            best_model_wts = copy.deepcopy(model.state_dict())
            no_improve_epochs = 0

            save_path = os.path.join(OUTPUT_DIR, save_name)
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_macro_f1": best_val_macro_f1,
                "class_names": class_names,
                "img_size": IMG_SIZE,
            }, save_path)
            print(f"\nSaved best model to: {save_path}")
        else:
            no_improve_epochs += 1
            print(f"\nNo validation improvement for {no_improve_epochs} epoch(s).")

        if no_improve_epochs >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_model_wts)
    return model, best_val_macro_f1, history

# =========================================================
# UNSEEN TEST SAMPLES
# =========================================================
def build_remapped_unseen_samples(unseen_test_dir, train_class_to_idx, mapping):
    unseen_test_dir = Path(unseen_test_dir)
    samples = []

    for test_folder_name, train_folder_name in mapping.items():
        folder = unseen_test_dir / test_folder_name

        if not folder.exists():
            raise FileNotFoundError(f"Missing unseen folder: {folder}")

        if train_folder_name not in train_class_to_idx:
            raise KeyError(f"Mapped label '{train_folder_name}' not found in training classes")

        label_idx = train_class_to_idx[train_folder_name]

        image_paths = []
        for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff", "*.webp"):
            image_paths.extend(folder.glob(ext))
            image_paths.extend(folder.glob(ext.upper()))

        for img_path in sorted(image_paths):
            samples.append((str(img_path), label_idx))

    return samples

# =========================================================
# MAIN
# =========================================================
def main():
    base_dataset = datasets.ImageFolder(TRAIN_DATASET_DIR)

    print("TRAIN classes:", base_dataset.classes)
    print("TRAIN class_to_idx:", base_dataset.class_to_idx)
    print("TRAIN total images:", len(base_dataset.samples))

    class_names = base_dataset.classes
    train_class_to_idx = base_dataset.class_to_idx

    # unseen test only for evaluation
    unseen_samples = build_remapped_unseen_samples(
        unseen_test_dir=UNSEEN_TEST_DIR,
        train_class_to_idx=train_class_to_idx,
        mapping=TEST_TO_TRAIN_LABEL_MAP
    )

    print("\nManual test label remap:")
    for k, v in TEST_TO_TRAIN_LABEL_MAP.items():
        print(f"  {k} -> {v}")
    print("UNSEEN TEST total images:", len(unseen_samples))

    all_samples = base_dataset.samples
    all_paths = [x[0] for x in all_samples]
    all_labels = [x[1] for x in all_samples]

    train_paths, val_paths, train_labels, val_labels = train_test_split(
        all_paths,
        all_labels,
        test_size=VAL_SPLIT,
        random_state=RANDOM_SEED,
        stratify=all_labels
    )

    train_samples = list(zip(train_paths, train_labels))
    val_samples = list(zip(val_paths, val_labels))

    train_dataset = KidneyUltrasoundDataset(
        samples=train_samples,
        transform=train_transform,
        apply_preprocessing=True
    )

    val_dataset = KidneyUltrasoundDataset(
        samples=val_samples,
        transform=eval_transform,
        apply_preprocessing=True
    )

    unseen_dataset = KidneyUltrasoundDataset(
        samples=unseen_samples,
        transform=eval_transform,
        apply_preprocessing=True
    )

    print("\nSplit summary:")
    print("Train size :", len(train_dataset))
    print("Val size   :", len(val_dataset))
    print("Test size  :", len(unseen_dataset))

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    unseen_loader = DataLoader(
        unseen_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    model = build_model(num_classes=len(class_names)).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_macro_f1": [],
        "val_bal_acc": [],
        "unseen_loss": [],
        "unseen_macro_f1": [],
        "unseen_bal_acc": [],
    }

    best_val_macro_f1 = 0.0

    # =====================================================
    # STAGE 1
    # =====================================================
    print("\n" + "=" * 80)
    print("STAGE 1: Train classifier only")
    print("=" * 80)

    freeze_all_features(model)
    print_trainable_params(model)

    optimizer_stage1 = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_STAGE1,
        weight_decay=WEIGHT_DECAY
    )

    scheduler_stage1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_stage1,
        mode="max",
        factor=0.5,
        patience=2
    )

    model, best_val_macro_f1, history = train_stage(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        unseen_loader=unseen_loader,
        optimizer=optimizer_stage1,
        scheduler=scheduler_stage1,
        criterion=criterion,
        class_names=class_names,
        epochs=EPOCHS_STAGE1,
        history=history,
        best_val_macro_f1=best_val_macro_f1,
        save_name="best_stage1_effb3.pth",
        stage_name="Stage 1"
    )

    # =====================================================
    # STAGE 2
    # =====================================================
    print("\n" + "=" * 80)
    print("STAGE 2: Unfreeze last 2 feature blocks")
    print("=" * 80)

    unfreeze_last_n_feature_blocks(model, n=2)
    print_trainable_params(model)

    optimizer_stage2 = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_STAGE2,
        weight_decay=WEIGHT_DECAY
    )

    scheduler_stage2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_stage2,
        mode="max",
        factor=0.5,
        patience=2
    )

    model, best_val_macro_f1, history = train_stage(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        unseen_loader=unseen_loader,
        optimizer=optimizer_stage2,
        scheduler=scheduler_stage2,
        criterion=criterion,
        class_names=class_names,
        epochs=EPOCHS_STAGE2,
        history=history,
        best_val_macro_f1=best_val_macro_f1,
        save_name="best_stage2_effb3.pth",
        stage_name="Stage 2"
    )

    # =====================================================
    # STAGE 3
    # =====================================================
    print("\n" + "=" * 80)
    print("STAGE 3: Fine-tune full model")
    print("=" * 80)

    unfreeze_all(model)
    print_trainable_params(model)

    optimizer_stage3 = torch.optim.AdamW(
        model.parameters(),
        lr=LR_STAGE3,
        weight_decay=WEIGHT_DECAY
    )

    scheduler_stage3 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_stage3,
        mode="max",
        factor=0.5,
        patience=2
    )

    model, best_val_macro_f1, history = train_stage(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        unseen_loader=unseen_loader,
        optimizer=optimizer_stage3,
        scheduler=scheduler_stage3,
        criterion=criterion,
        class_names=class_names,
        epochs=EPOCHS_STAGE3,
        history=history,
        best_val_macro_f1=best_val_macro_f1,
        save_name="best_stage3_effb3.pth",
        stage_name="Stage 3"
    )

    final_model_path = os.path.join(OUTPUT_DIR, "final_effb3_kidney_ultrasound.pth")
    torch.save({
        "model_state_dict": model.state_dict(),
        "class_names": class_names,
        "img_size": IMG_SIZE,
        "best_val_macro_f1": best_val_macro_f1,
    }, final_model_path)

    print(f"\nFinal model saved to: {final_model_path}")
    print(f"Best Validation Macro F1: {best_val_macro_f1:.4f}")

    final_val_metrics = evaluate_model(
        model=model,
        loader=val_loader,
        criterion=criterion,
        class_names=class_names,
        save_csv_path=os.path.join(OUTPUT_DIR, "final_val_misclassified.csv")
    )
    print_metrics("FINAL VALIDATION RESULTS", final_val_metrics, class_names)

    final_unseen_metrics = evaluate_model(
        model=model,
        loader=unseen_loader,
        criterion=criterion,
        class_names=class_names,
        save_csv_path=os.path.join(OUTPUT_DIR, "final_unseen_misclassified.csv")
    )
    print_metrics("FINAL UNSEEN TEST RESULTS", final_unseen_metrics, class_names)

if __name__ == "__main__":
    main()

TRAIN classes: ['defected', 'normal']
TRAIN class_to_idx: {'defected': 0, 'normal': 1}
TRAIN total images: 10000

Manual test label remap:
  Diseased -> defected
  Normal -> normal
UNSEEN TEST total images: 20

Split summary:
Train size : 8000
Val size   : 2000
Test size  : 20

STAGE 1: Train classifier only
Trainable params: 393,986 / 11,090,218 (3.55%)

Stage 1 | Epoch [1/5]
--------------------------------------------------------------------------------


Stage 1 Train 1:   0%|          | 0/500 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/125 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

Train Loss         : 0.5403

Validation:
  Loss             : 0.4496
  Accuracy         : 0.7915
  Balanced Acc     : 0.7915
  Macro F1         : 0.7900

Unseen Benchmark Test:
  Loss             : 1.0441
  Accuracy         : 0.5000
  Balanced Acc     : 0.4792
  Macro F1         : 0.4792

Validation Confusion Matrix:
[[877 123]
 [294 706]]

Unseen Test Confusion Matrix:
[[7 5]
 [5 3]]

Saved best model to: /kaggle/working/effb3_kidney_ultrasound/best_stage1_effb3.pth

Stage 1 | Epoch [2/5]
--------------------------------------------------------------------------------


Stage 1 Train 2:   0%|          | 0/500 [00:00<?, ?it/s]

... (47 outputs left out)

Evaluating:   0%|          | 0/125 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

Train Loss         : 0.3015

Validation:
  Loss             : 0.6638
  Accuracy         : 0.6940
  Balanced Acc     : 0.6940
  Macro F1         : 0.6899

Unseen Benchmark Test:
  Loss             : 1.1869
  Accuracy         : 0.5500
  Balanced Acc     : 0.5000
  Macro F1         : 0.4872

Validation Confusion Matrix:
[[809 191]
 [421 579]]

Unseen Test Confusion Matrix:
[[9 3]
 [6 2]]

No validation improvement for 1 epoch(s).

Stage 2 | Epoch [8/8]
--------------------------------------------------------------------------------


Stage 2 Train 8:   0%|          | 0/500 [00:00<?, ?it/s]

In [ ]:
print("hellow")

In [ ]:
Dataset_multicalasess

In [28]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from oauth2client.client import GoogleCredentials
from google.colab import auth

# Authenticate
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

folder_id = "12Rpd816XFV67aLk_cS4acxqadW1j4kY-"

file = drive.CreateFile({
    'title': 'Test_unseen.zip',
    'parents': [{'id': folder_id}]
})

file.SetContentFile('Test_unseen.zip')
file.Upload()

print("✅ ZIP uploaded successfully to your Google Drive folder")

✅ ZIP uploaded successfully to your Google Drive folder


In [2]:
!pip install -q PyDrive2

In [3]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from oauth2client.client import GoogleCredentials
from google.colab import auth

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
Test_unseen

In [27]:
!zip -r Test_unseen.zip '/kaggle/input/datasets/ahmedshayan1/test-unseen'

  adding: kaggle/input/datasets/ahmedshayan1/test-unseen/ (stored 0%)
  adding: kaggle/input/datasets/ahmedshayan1/test-unseen/Test/ (stored 0%)
  adding: kaggle/input/datasets/ahmedshayan1/test-unseen/Test/Diseased/ (stored 0%)
  adding: kaggle/input/datasets/ahmedshayan1/test-unseen/Test/Diseased/lbox_38978-Afbeelding1 (1).jpg (deflated 5%)
  adding: kaggle/input/datasets/ahmedshayan1/test-unseen/Test/Diseased/1-s2.0-S0720048X03000731-gr1.jpg (deflated 0%)
  adding: kaggle/input/datasets/ahmedshayan1/test-unseen/Test/Diseased/lbox_38978-Afbeelding1.jpg (deflated 5%)
  adding: kaggle/input/datasets/ahmedshayan1/test-unseen/Test/Diseased/lbox_64246-Afbeelding1.jpg (deflated 5%)
  adding: kaggle/input/datasets/ahmedshayan1/test-unseen/Test/Diseased/SnapShot151-300x169.jpg (deflated 7%)
  adding: kaggle/input/datasets/ahmedshayan1/test-unseen/Test/Diseased/Fig6Ultrasound.jpg (deflated 12%)
  adding: kaggle/input/datasets/ahmedshayan1/test-unseen/Test/Diseased/1.jpeg (deflated 4%)
  addin

In [19]:
import os

folder_path = '/kaggle/input/datasets/ahmedshayan1/dataset-3/Dataset_3'

for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_path = os.path.join(root, file)
        
        print("Uploading:", file_path)  # debug line
        
        gfile = drive.CreateFile({'title': file})
        gfile.SetContentFile(file_path)
        gfile.Upload()

        print(f'Uploaded: {file}')

In [18]:
# import os

# for root, dirs, files in os.walk('/kaggle/input/datasets/ahmedshayan1/dataset-3/Dataset_3'):
#     print("Folder:", root)
#     print("Files:", files)
#     print("-" * 40)